# Sentiment Analysis

This notebook applies sentiment analysis to the cleaned YouTube comment dataset.

The analysis begins with VADER as a baseline sentiment model and examines sentiment polarity across the three content domains.

The workflow includes:
- generating sentiment scores
- assigning sentiment categories
- examining the overall sentiment distribution
- comparing sentiment across domains
- examining the relationship between sentiment and engagement

## Important Methodological Note

VADER classifications are model outputs, not ground-truth sentiment labels. The percentages and comparisons in this notebook therefore describe **VADER's classifications of the sampled comments**.

A manually annotated validation sample is created later in this notebook so that model performance can be evaluated against human judgments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
input_path = Path("../data/processed/youtube_comments_clean.csv")

df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
df.head()

## VADER Sentiment Scoring

VADER (Valence Aware Dictionary and sEntiment Reasoner) is used as the baseline sentiment model.

It is particularly suitable for short, informal text because it considers features such as punctuation, capitalization, and common sentiment expressions.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

In [ ]:
sample_comments = df["clean_text"].sample(10, random_state=42)

for comment in sample_comments:
    scores = analyzer.polarity_scores(comment)
    print(f"Comment: {comment}")
    print(f"Scores: {scores}")
    print()

In [ ]:
sentiment_scores = df["clean_text"].apply(analyzer.polarity_scores)
sentiment_scores = pd.DataFrame(sentiment_scores.tolist())

sentiment_scores = sentiment_scores.rename(
    columns={
        "neg": "vader_neg",
        "neu": "vader_neu",
        "pos": "vader_pos",
        "compound": "vader_compound"
    }
)

df = pd.concat([df, sentiment_scores], axis=1)

df.head()

In [ ]:
df[["vader_neg", "vader_neu", "vader_pos", "vader_compound"]].describe()

In [ ]:
def classify_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"


df["vader_sentiment"] = df["vader_compound"].apply(classify_sentiment)

df["vader_sentiment"].value_counts()

In [ ]:
sentiment_summary = (
    df["vader_sentiment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

sentiment_summary

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="vader_sentiment",
    order=["positive", "neutral", "negative"]
)

plt.title("Overall Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Comments")

plt.tight_layout()
plt.show()

## Sentiment Distribution by Domain

The overall sentiment distribution can hide differences between content domains.

Sentiment categories are therefore compared across technology, travel, and cats using both counts and percentages.

In [ ]:
domain_sentiment_counts = pd.crosstab(
    df["domain"],
    df["vader_sentiment"]
)[["positive", "neutral", "negative"]]

domain_sentiment_counts

In [ ]:
domain_sentiment_pct = (
    pd.crosstab(
        df["domain"],
        df["vader_sentiment"],
        normalize="index"
    )[["positive", "neutral", "negative"]]
    .mul(100)
    .round(2)
)

domain_sentiment_pct

In [ ]:
domain_sentiment_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5)
)

plt.title("Sentiment Distribution by Domain")
plt.xlabel("Domain")
plt.ylabel("Percentage of Comments")
plt.xticks(rotation=0)
plt.legend(title="Sentiment")

plt.tight_layout()
plt.show()

### Observation

VADER sentiment classifications differ across the three sampled domains.

VADER classified the cats sample with the highest proportion of positive comments (72.8%), while technology had the highest proportion of negative comments (25.6%). Travel had 50.0% positive, 39.2% neutral, and 10.8% negative comments.

These differences describe the sampled comments rather than the broader audiences for each content domain. The results may also reflect differences in individual creators, video topics, and the specific videos selected.

## Sentiment Intensity by Domain

Sentiment categories provide a simplified view of polarity. The VADER compound score provides additional information about the strength and direction of sentiment.

The distribution of compound scores is therefore compared across the three domains.

In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df,
    x="domain",
    y="vader_compound"
)

plt.axhline(0, linewidth=1)

plt.title("VADER Compound Sentiment by Domain")
plt.xlabel("Domain")
plt.ylabel("Compound Sentiment Score")

plt.tight_layout()
plt.show()

### Observation

The VADER compound scores show differences in sentiment intensity across the sampled domains. The cats comments had the highest mean (0.434) and median (0.511) compound scores, while technology had the lowest mean (0.117) and a median of 0.000. Travel fell between the two, with a mean of 0.260 and median of 0.039.

All three domains contained both positive and negative sentiment, indicating variation within each domain. These results describe the selected comments and should not be generalized to the broader audiences without additional sampling.

In [ ]:
sentiment_intensity = (
    df.groupby("domain")["vader_compound"]
      .agg(["mean", "median", "std", "min", "max"])
      .round(3)
)

sentiment_intensity

## Sentiment and Audience Engagement

This section examines whether comments with different sentiment scores receive different levels of engagement.

Because comment likes are highly skewed, the analysis uses log-transformed likes. The relationship is examined descriptively rather than interpreted as causal.

In [ ]:
engagement_by_sentiment = (
    df.groupby("vader_sentiment")["like_count"]
      .agg(["count", "mean", "median", "max"])
      .round(2)
)

engagement_by_sentiment

In [ ]:
df["log_like_count"] = np.log1p(df["like_count"])

In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df,
    x="vader_sentiment",
    y="log_like_count",
    order=["positive", "neutral", "negative"]
)

plt.title("Comment Engagement by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("log(1 + likes)")

plt.tight_layout()
plt.show()

In [ ]:
engagement_log_summary = (
    df.groupby("vader_sentiment")["log_like_count"]
      .agg(["mean", "median", "max"])
      .round(3)
)

engagement_log_summary

In [ ]:
nonzero_engagement = (
    df.assign(has_like=df["like_count"] > 0)
      .groupby("vader_sentiment")["has_like"]
      .mean()
      .mul(100)
      .round(2)
)

nonzero_engagement

In [ ]:
sentiment_engagement_corr = df["vader_compound"].corr(
    df["like_count"],
    method="spearman"
)

print("Spearman correlation:", round(sentiment_engagement_corr, 3))

### Observation

VADER sentiment intensity showed a very weak positive association with comment likes (Spearman ρ = 0.092). The proportion of comments receiving at least one like was 11.95% for positive comments, 8.77% for neutral comments, and 7.14% for negative comments.

However, most comments received no likes, and a small number of highly liked comments created substantial skew in the engagement data. Therefore, the relationship between sentiment and engagement appears weak in this sample and should not be interpreted as causal.

## VADER Sentiment Examples

Examining individual comments helps identify how VADER handles clear positive and negative language, emojis, informal expressions, humor, and context-dependent comments.

Comments with compound scores close to zero are also examined because they represent comments with little overall polarity according to VADER.

In [ ]:
top_positive = (
    df.sort_values("vader_compound", ascending=False)
      [["domain", "video_title", "clean_text", "vader_compound"]]
      .head(10)
)

top_positive

In [ ]:
top_negative = (
    df.sort_values("vader_compound", ascending=True)
      [["domain", "video_title", "clean_text", "vader_compound"]]
      .head(10)
)
top_negative

In [ ]:
closest_to_neutral = (
    df.assign(abs_compound=df["vader_compound"].abs())
      .sort_values("abs_compound")
      [["domain", "video_title", "clean_text", "vader_compound"]]
      .head(15)
)

closest_to_neutral

## Sentiment Distribution by Video

Domain-level sentiment can hide variation between individual videos. This section examines sentiment composition at the video level.

In [ ]:
video_sentiment_pct = (
    pd.crosstab(
        df["video_title"],
        df["vader_sentiment"],
        normalize="index"
    )[["positive", "neutral", "negative"]]
    .mul(100)
    .round(2)
)

video_sentiment_pct

In [ ]:
video_sentiment_pct.plot(
    kind="barh",
    stacked=True,
    figsize=(12, 8)
)

plt.title("Sentiment Distribution by Video")
plt.xlabel("Percentage of Comments")
plt.ylabel("Video")

plt.legend(title="Sentiment")

plt.tight_layout()
plt.show()

### Observation

Sentiment composition varies substantially across individual videos, even within the same domain. This indicates that domain-level differences may partly reflect the topics and characteristics of the selected videos rather than the content domain alone.

Therefore, the domain-level sentiment results should be interpreted as descriptive findings from the selected videos rather than as general characteristics of technology, travel, or cat audiences.

## Human Validation Sample

VADER provides model-generated sentiment classifications; these labels should not be treated as ground truth.

A stratified sample of 120 comments is prepared for manual annotation:
- 40 technology comments
- 40 travel comments
- 40 cats comments

The VADER label is intentionally excluded from the annotation file so that the manual labels are not influenced by the model output.

In [ ]:
validation_sample = (
    df.groupby("domain", group_keys=False)
      .sample(n=40, random_state=42)
      [["comment_id", "domain", "creator", "video_id", "video_title", "clean_text"]]
      .copy()
)

validation_sample["human_sentiment"] = ""

validation_path = Path("../data/processed/vader_validation_sample.csv")
validation_sample.to_csv(validation_path, index=False)

print("Validation sample shape:", validation_sample.shape)
print("Comments per domain:")
print(validation_sample["domain"].value_counts())
print(f"Saved annotation file to: {validation_path}")

### Manual Annotation

Open `data/processed/vader_validation_sample.csv` and fill the `human_sentiment` column using:

- `positive`
- `neutral`
- `negative`

The manual labels will later be compared with VADER and a transformer model to evaluate model performance.

In [ ]:
output_path = Path("../data/processed/youtube_comments_vader.csv")

df.to_csv(output_path, index=False)

print(f"Saved VADER-enriched dataset to: {output_path}")

In [ ]:
vader_df = pd.read_csv(output_path)

print("Saved dataset shape:", vader_df.shape)
print("Columns:")
print(vader_df.columns.tolist())